In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==========================================================
# Configuration
# ==========================================================

BRONZE_TABLE = "finance_intelligence_hub.bronze.companies"
SILVER_TABLE = "finance_intelligence_hub.silver.companies"
QUARANTINE_TABLE = "finance_intelligence_hub.silver_rejected.companies"

PIPELINE_NAME = "Companies Silver Pipeline"
PIPELINE_VERSION = "1.1"

print("="*80)
print(PIPELINE_NAME)
print("="*80)

# ==========================================================
# Helper Functions
# ==========================================================

def safe_double(column):

    c = F.trim(column)
    c = F.when(
        c.rlike(r"^(-+|N/A|n/a|NaN|None|null)?$"),
        F.lit(None)
    ).otherwise(c)

    return c.try_cast("double")


def clean_percentage(column):

    cleaned = F.regexp_replace(column, "%", "")
    return safe_double(cleaned)


def clean_pe_ratio(column):

    return safe_double(column)


def clean_large_number(column):
    
    return (
        F.when(
            column.endswith("K"),
            safe_double(F.regexp_replace(column, "K", "")) * 1_000
        )
        .when(
            column.endswith("M"),
            safe_double(F.regexp_replace(column, "M", "")) * 1_000_000
        )
        .when(
            column.endswith("B"),
            safe_double(F.regexp_replace(column, "B", "")) * 1_000_000_000
        )
        .when(
            column.endswith("T"),
            safe_double(F.regexp_replace(column, "T", "")) * 1_000_000_000_000
        )
        .otherwise(
            safe_double(column)
        )
    )


def split_52_week(column):
    """
    52_wk_range بيجي كنص فيه رقمين متفصلين بمسافة:
    "45.20 89.10" -> low=45.20 , high=89.10
    لو الصيغة مش زي كده (فاضي / رقم واحد بس) -> NULL, NULL
    """
    parts = F.split(F.trim(column), r"\s+")

    low = F.when(
        F.size(parts) == 2,
        safe_double(parts.getItem(0))
    ).otherwise(F.lit(None).cast("double"))

    high = F.when(
        F.size(parts) == 2,
        safe_double(parts.getItem(1))
    ).otherwise(F.lit(None).cast("double"))

    return low, high


# ==========================================================
# Create Silver Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {SILVER_TABLE}

(

ticker STRING,

company_name STRING,

category STRING,

price DOUBLE,

change DOUBLE,

change_percent DOUBLE,

volume BIGINT,

avg_volume_3m BIGINT,

market_cap DOUBLE,

pe_ratio DOUBLE,

week52_change_percent DOUBLE,

week52_low DOUBLE,

week52_high DOUBLE,

dwh_loaded_at TIMESTAMP,

silver_loaded_at TIMESTAMP,

pipeline_name STRING,

pipeline_version STRING

)

USING DELTA

""")

print("Silver table verified successfully.")


# ==========================================================
# Incremental Load
# ==========================================================

last_loaded = spark.sql(f"""

SELECT

COALESCE(

MAX(dwh_loaded_at),

TIMESTAMP('1900-01-01')

)

FROM {SILVER_TABLE}

""").first()[0]

bronze_df = (

spark.table(BRONZE_TABLE)

.filter(

F.col("dwh_loaded_at") > last_loaded

)

)

# ==========================================================
# Global Numeric Text Cleanup
# Remove thousand separators from all numeric-like columns
# ==========================================================

numeric_string_columns = [
    "price",
    "change",
    "volume",
    "avg_volume_3m",
    "market_cap",
    "PE_ratio",
    "change_percent",
    "52_wk_change_percent",
    "52_wk_range"
]

for col_name in numeric_string_columns:

    bronze_df = bronze_df.withColumn(
        col_name,
        F.regexp_replace(F.col(col_name), ",", "")
    )

print("Numeric text cleanup completed.")

rows_read = bronze_df.count()

print(f"Rows Read : {rows_read}")

if rows_read == 0:

    dbutils.notebook.exit("No new records.")

# ==========================================================
# Data Cleaning
# ==========================================================

week52_low_col, week52_high_col = split_52_week(F.col("52_wk_range"))

clean_df = (

    bronze_df

    #########################################################
    # Basic Cleaning
    #########################################################

    .withColumn(
        "ticker",
        F.upper(F.trim("ticker"))
    )

    .withColumn(
        "company_name",
        F.trim("company_name")
    )

    .withColumn(
        "category",
        F.trim("category")
    )

    #########################################################
    # Prices
    #########################################################

    .withColumn(
        "price",
        safe_double(F.col("price"))
    )

    .withColumn(
        "change",
        safe_double(F.col("change"))
    )

    #########################################################
    # Percentages
    #########################################################

    .withColumn(
        "change_percent",
        clean_percentage(F.col("change_percent"))
    )

    .withColumn(
        "week52_change_percent",
        clean_percentage(
            F.col("52_wk_change_percent")
        )
    )

    #########################################################
    # Large Numbers
    #########################################################

    .withColumn(
        "volume",
        clean_large_number(
            F.col("volume")
        ).cast("bigint")
    )

    .withColumn(
        "avg_volume_3m",
        clean_large_number(
            F.col("avg_volume_3m")
        ).cast("bigint")
    )

    .withColumn(
        "market_cap",
        clean_large_number(
            F.col("market_cap")
        )
    )

    #########################################################
    # PE Ratio
    #########################################################

    .withColumn(
        "pe_ratio",
        clean_pe_ratio(
            F.col("PE_ratio")
        )
    )

    #########################################################
    # 52 Week Range
    #########################################################

    .withColumn(
        "week52_low",
        week52_low_col
    )

    .withColumn(
        "week52_high",
        week52_high_col
    )

    #########################################################
    # Audit
    #########################################################

    .withColumn(
        "silver_loaded_at",
        F.current_timestamp()
    )

    .withColumn(
        "pipeline_name",
        F.lit(PIPELINE_NAME)
    )

    .withColumn(
        "pipeline_version",
        F.lit(PIPELINE_VERSION)
    )

)

print(f"Rows after cleaning : {clean_df.count()}")

# ==========================================================
# Create Quarantine Table
# ==========================================================

if not spark.catalog.tableExists(QUARANTINE_TABLE):

    (
        clean_df.limit(0)

        .withColumn("reject_reason", F.lit(None).cast("string"))
        .withColumn("rejected_at", F.lit(None).cast("timestamp"))

        .write
        .format("delta")
        .saveAsTable(QUARANTINE_TABLE)
    )

print("Quarantine table verified successfully.")

# ==========================================================
# Business Rules
# ==========================================================

valid_df = clean_df.filter(

    (F.col("ticker").isNotNull()) &
    (F.length("ticker") > 0) &

    (F.col("company_name").isNotNull()) &

    (F.col("price") > 0) &

    (F.col("market_cap") > 0) &

    (F.col("volume") >= 0) &

    (F.col("avg_volume_3m") >= 0) &

    (F.col("week52_low") > 0) &

    (F.col("week52_high") > 0) &

    (F.col("week52_high") >= F.col("week52_low"))

)

rejected_df = clean_df.subtract(valid_df)

rejected_df = (

    rejected_df

    .withColumn(
        "reject_reason",
        F.lit("Failed Business Validation")
    )

    .withColumn(
        "rejected_at",
        F.current_timestamp()
    )

)

rows_rejected = rejected_df.count()

print(f"Rejected Rows : {rows_rejected}")

if rows_rejected > 0:

    (

        rejected_df.write

        .format("delta")

        .mode("append")

        .option("mergeSchema", "true")

        .saveAsTable(QUARANTINE_TABLE)

    )

# ==========================================================
# Remove Duplicates
# ==========================================================

window_spec = (

    Window

    .partitionBy("ticker")

    .orderBy(
        F.col("dwh_loaded_at").desc()
    )

)

silver_df = (

    valid_df

    .withColumn(
        "rn",
        F.row_number().over(window_spec)
    )

    .filter(
        F.col("rn") == 1
    )

    .drop("rn")

)

rows_valid = silver_df.count()

print(f"Valid Rows : {rows_valid}")

silver_df.printSchema()

silver_df.show(10, truncate=False)


# ==========================================================
# Merge Into Silver
# ==========================================================

print("=" * 80)
print("Starting MERGE...")
print("=" * 80)

delta_table = DeltaTable.forName(
    spark,
    SILVER_TABLE
)

(
    delta_table.alias("target")

    .merge(

        silver_df.alias("source"),

        """
        target.ticker = source.ticker
        """

    )

    .whenMatchedUpdate(

        condition="""
        source.dwh_loaded_at > target.dwh_loaded_at
        """,

        set={

            "company_name": "source.company_name",

            "category": "source.category",

            "price": "source.price",

            "change": "source.change",

            "change_percent": "source.change_percent",

            "volume": "source.volume",

            "avg_volume_3m": "source.avg_volume_3m",

            "market_cap": "source.market_cap",

            "pe_ratio": "source.pe_ratio",

            "week52_change_percent":
                "source.week52_change_percent",

            "week52_low":
                "source.week52_low",

            "week52_high":
                "source.week52_high",

            "dwh_loaded_at":
                "source.dwh_loaded_at",

            "silver_loaded_at":
                "source.silver_loaded_at",

            "pipeline_name":
                "source.pipeline_name",

            "pipeline_version":
                "source.pipeline_version"

        }

    )

    .whenNotMatchedInsert(

        values={

            "ticker": "source.ticker",

            "company_name": "source.company_name",

            "category": "source.category",

            "price": "source.price",

            "change": "source.change",

            "change_percent": "source.change_percent",

            "volume": "source.volume",

            "avg_volume_3m": "source.avg_volume_3m",

            "market_cap": "source.market_cap",

            "pe_ratio": "source.pe_ratio",

            "week52_change_percent":
                "source.week52_change_percent",

            "week52_low":
                "source.week52_low",

            "week52_high":
                "source.week52_high",

            "dwh_loaded_at":
                "source.dwh_loaded_at",

            "silver_loaded_at":
                "source.silver_loaded_at",

            "pipeline_name":
                "source.pipeline_name",

            "pipeline_version":
                "source.pipeline_version"

        }

    )

    .execute()

)

print("MERGE completed successfully.")

# ==========================================================
# Optimize Delta Table
# ==========================================================

print("Running OPTIMIZE...")

spark.sql(f"""

OPTIMIZE {SILVER_TABLE}

ZORDER BY (ticker)

""")

print("Optimization completed.")

# ==========================================================
# Pipeline Metrics
# ==========================================================

total_rows = spark.sql(f"""

SELECT COUNT(*)

FROM {SILVER_TABLE}

""").first()[0]

total_companies = spark.sql(f"""

SELECT COUNT(DISTINCT ticker)

FROM {SILVER_TABLE}

""").first()[0]

print()

print("="*80)

print("Pipeline Summary")

print("="*80)

print(f"Rows Read            : {rows_read:,}")

print(f"Rows Valid           : {rows_valid:,}")

print(f"Rows Rejected        : {rows_rejected:,}")

print(f"Companies in Silver  : {total_companies:,}")

print(f"Total Silver Rows    : {total_rows:,}")

print("="*80)

# ==========================================================
# Delta History
# ==========================================================

spark.sql(f"""

DESCRIBE HISTORY {SILVER_TABLE}

""").show(truncate=False)